**Project:** Analyzing the Impact of Social Media Sentiment on
Investor Herding Behavior in Financial Markets Using Agent-Based Modeling

**Course:** FM 4054 — Agent-Based Modeling | University of Colombo

# **Mount Google Drive**

This cell connects the Colab session to Google Drive, allowing the
notebook to read input files and save output files persistently.
Without this step, all files generated during the session would be
lost when the runtime disconnects.

**Action required:** Click the link, sign in with your Google account,
and paste the authorisation code when prompted.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Import Required Libraries**

This cell imports all Python libraries used throughout the pipeline.
Running this cell before any other processing step ensures that all
functions and modules are available in memory for subsequent cells.

In [22]:
import os
import pandas as pd
import re
import yfinance as yf
import torch
import gc
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification


print("Libraries imported successfully!")

Libraries imported successfully!


# **Set project path**

This cell sets the working directory to the project folder in Google
Drive.

In [23]:
# Enter the project path
PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/crypto_abm_project'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f"Directory successfully changed to: {os.getcwd()}")
else:
    print(f"ERROR: The path '{PROJECT_PATH}' does not exist. Please verify the folder name in Google Drive.")

Directory successfully changed to: /content/drive/MyDrive/Colab Notebooks/crypto_abm_project


# **Create Output Folders**

This cell creates the `data/` and `outputs/` folders inside the
project directory if they do not already exist. It also verifies
that the raw tweet dataset has been uploaded to the correct location.

**Expected folder structure after this cell:**

crypto_abm_project/
  - ├── data/
    - └── Merged_Twitter_Data_IDsRemoved.csv
  - └── outputs/
    - └── (all generated files will be saved here)

In [24]:
os.makedirs('outputs', exist_ok=True)
os.makedirs('data',    exist_ok=True)
print("Folders ready.")

Folders ready.


# **Clean the Tweet Data**

Loads the raw tweet dataset and cleans each tweet before sentiment analysis.

**Cleaning steps applied to every tweet:**
1. Remove URLs (`http://...`, `https://...`)
2. Remove @mentions (e.g. `@elonmusk`)
3. Keep hashtag words but remove the `#` symbol (e.g. `#Bitcoin` → `bitcoin`)
4. Remove punctuation and special characters
5. Normalise whitespace and convert to lowercase
6. Drop tweets shorter than 15 characters after cleaning (too short to be meaningful)

An **engagement weight** (`likes + retweets + 1`) is also added here.
The `+1` ensures tweets with zero engagement still count — they represent the long tail of retail opinion.

**Expected output:** `outputs/tweets_cleaned.csv`

In [25]:
df = pd.read_csv(
    'data/Merged_Twitter_Data_IDsRemoved.csv',
    encoding='latin-1'
)

# Parse dates
df['date'] = pd.to_datetime(
    df['date'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

# Remove rows with missing essential information
df = df.dropna(subset=['date', 'text']).copy()

# Ensure engagement columns are numeric
for column in ['likes', 'retweets', 'followers']:
    df[column] = pd.to_numeric(
        df[column],
        errors='coerce'
    ).fillna(0)

    # Prevent negative engagement values
    df[column] = df[column].clip(lower=0)

# Sort observations by date
df = df.sort_values('date').reset_index(drop=True)

# Clean tweet text
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)        # keep hashtag words, remove #
    text = re.sub(r'[^\w\s]', '', text)          # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()     # remove extra spaces
    return text.lower()

df['text_clean'] = df['text'].apply(clean_tweet)

# Drop tweets that are too short after cleaning
df = df[
    df['text_clean'].str.len() > 15
].reset_index(drop=True)

# Remove duplicate tweets after text cleaning
rows_before = len(df)

df = df.drop_duplicates(
    subset=['date', 'text_clean'],
    keep='first'
).reset_index(drop=True)

duplicates_removed = rows_before - len(df)

# Engagement weight for later aggregation
df['weight'] = df['likes'] + df['retweets'] + 1

print(f"Duplicate tweets removed: {duplicates_removed:,}")
print(f"Cleaned tweets:           {len(df):,}")
print(f"Unique dates:             {df['date'].dt.date.nunique()}")

df.to_csv(
    'outputs/tweets_cleaned.csv',
    index=False
)

print("Saved → outputs/tweets_cleaned.csv")

Duplicate tweets removed: 25,977
Cleaned tweets:           80,877
Unique dates:             36
Saved → outputs/tweets_cleaned.csv


# **FinBERT Sentiment Scoring**

This is the core NLP step of the pipeline. It applies the
**FinBERT** model (`ProsusAI/finbert`) — a BERT-based language
model pre-trained specifically on financial text — to every cleaned
tweet to extract a sentiment score.

**How the score is calculated:**

FinBERT outputs three class probabilities per tweet:
- P(positive), P(negative), P(neutral)

The per-tweet sentiment score is computed as:

$$S = P(\text{positive}) - P(\text{negative}) \in [-1, 1]$$

A score of +1 indicates strongly positive sentiment; -1 indicates
strongly negative.

**Daily aggregation** uses an engagement-weighted mean so that
high-reach, viral tweets carry proportionally more influence
than low-engagement posts:

$$S_t = \frac{\sum_i w_i \cdot s_i}{\sum_i w_i}, \quad w_i = \text{likes}_i + \text{retweets}_i + 1$$

**Processing note:** A batch size of 256 is used to maximise
throughput on the Colab T4 GPU. Progress is saved as a checkpoint
every 5,120 tweets in case the session disconnects.

**Outputs:**
- `outputs/tweets_scored.csv` — all tweets with individual FinBERT scores
- `outputs/daily_sentiment.csv` — 36 rows, one weighted $S_t$ per date

In [26]:
# Clear any cached GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load FinBERT
print("Loading FinBERT...")
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(device)
model.eval()

# Load cleaned tweets
df = pd.read_csv('outputs/tweets_cleaned.csv', parse_dates=['date'])
print(f"Scoring {len(df):,} tweets with FinBERT...")

# Reduced batch size to prevent CUDA OOM
BATCH = 256
all_scores = []

# Process in smaller batches
for start in range(0, len(df), BATCH):
    end = min(start + BATCH, len(df))
    batch_texts = df['text_clean'].iloc[start:end].tolist()

    # Tokenize batch with dynamic padding to save memory
    inputs = tokenizer(
        batch_texts,
        max_length=512,
        return_tensors='pt',
        truncation=True,
        padding=True
    ).to(device)

    # Predict sentiment
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1)
    # FinBERT label order: positive=0, negative=1, neutral=2
    scores = (probs[:, 0] - probs[:, 1]).cpu().tolist()
    all_scores.extend(scores)

    # Print progress every 5,120 rows to keep logs clean
    if end % 5120 == 0 or end == len(df):
        print(f"  {end:,} / {len(df):,} done")

        # Save checkpoint periodically
        temp = df.iloc[:end].copy()
        temp['finbert_score'] = all_scores
        temp.to_csv('outputs/tweets_scored_checkpoint.csv', index=False)

df['finbert_score'] = all_scores
df.to_csv('outputs/tweets_scored.csv', index=False)
print("Saved → outputs/tweets_scored.csv")

# Aggregate to daily St (Weighted mean)
def weighted_mean_sentiment(group):
    return (group['finbert_score'] * group['weight']).sum() / group['weight'].sum()

daily = df.groupby(df['date'].dt.date).apply(weighted_mean_sentiment).reset_index()
daily.columns = ['date', 'St']
daily['date'] = pd.to_datetime(daily['date'])
daily['tick'] = range(1, len(daily) + 1)

# Get daily counts
counts = df.groupby(df['date'].dt.date).size().reset_index()
counts.columns = ['date', 'tweet_count']
counts['date'] = pd.to_datetime(counts['date'])
daily = daily.merge(counts, on='date')

daily.to_csv('outputs/daily_sentiment.csv', index=False)

print("\nDaily sentiment scores calculated successfully!")
print(daily[['date', 'tweet_count', 'St']].to_string(index=False))

Using device: cuda
Loading FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Scoring 80,877 tweets with FinBERT...
  5,120 / 80,877 done
  10,240 / 80,877 done
  15,360 / 80,877 done
  20,480 / 80,877 done
  25,600 / 80,877 done
  30,720 / 80,877 done
  35,840 / 80,877 done
  40,960 / 80,877 done
  46,080 / 80,877 done
  51,200 / 80,877 done
  56,320 / 80,877 done
  61,440 / 80,877 done
  66,560 / 80,877 done
  71,680 / 80,877 done
  76,800 / 80,877 done
  80,877 / 80,877 done
Saved → outputs/tweets_scored.csv

Daily sentiment scores calculated successfully!
      date  tweet_count        St
2022-06-08         2651  0.008557
2022-06-17         2513 -0.042043
2022-06-29         2004 -0.001217
2022-07-11         2234 -0.064131
2022-07-19         2092  0.031695
2022-07-27         2304  0.134556
2022-08-02         2422 -0.069805
2022-08-09         2354  0.017999
2022-08-19         1756  0.001132
2022-09-05         1561  0.041133
2022-09-11         1166  0.055493
2022-09-23         2085  0.099090
2022-10-09         1165  0.020161
2022-10-21         2084  0.056775
20

# **Get Real Bitcoin Prices (for validation)**

This cell downloads daily Bitcoin (BTC-USD) closing prices for the
same date range as the tweet dataset (June 2022 – May 2023) using
the `yfinance` library.

**Why Bitcoin?**
Although 80% of tweets use the word "crypto" generically, Bitcoin
is used as the validation benchmark because it functions as the
primary market indicator for the entire cryptocurrency space.
Sentiment-driven herding behaviour in the broader market is most
visibly reflected in Bitcoin price movements. This approach is
consistent with Bokhari (2025), one of the reviewed articles.

The daily sentiment scores and BTC prices are merged on the 36
available tweet dates to produce a single consolidated file used
for model validation in later analysis.

**Output:** `outputs/daily_sentiment_with_btc.csv` (36 rows)

In [28]:
# Load the daily sentiment data calculated in the previous step
daily = pd.read_csv('outputs/daily_sentiment.csv', parse_dates=['date'])

# Download BTC price for the full project period
print("Downloading BTC price data via yfinance...")
btc = yf.download(
    'BTC-USD',
    start='2022-06-01',
    end='2023-06-01',
    auto_adjust=True ,
    progress=False
)[['Close']]

# Reset index and clean column names
btc.index = pd.to_datetime(btc.index)
btc.columns = ['btc_close']
btc = btc.reset_index()
btc.rename(columns={'Date': 'date'}, inplace=True)
btc['date'] = pd.to_datetime(btc['date']).dt.tz_localize(None)

# Merge BTC prices with your daily sentiment dataset
daily['date'] = pd.to_datetime(daily['date'])
merged = daily.merge(btc, on='date', how='left')

# Forward-fill any missing prices just in case
merged['btc_close'] = merged['btc_close'].ffill()

# Save the final consolidated dataset
merged.to_csv('outputs/daily_sentiment_with_btc.csv', index=False)
print("Saved → outputs/daily_sentiment_with_btc.csv")

print("\nFinal Merged Data (Sentiment + BTC Prices):")
print(merged[['date', 'St', 'btc_close']].to_string(index=False))

Saved → outputs/daily_sentiment_with_btc.csv

Final Merged Data (Sentiment + BTC Prices):
      date        St    btc_close
2022-06-08  0.008557 30214.355469
2022-06-17 -0.042043 20471.482422
2022-06-29 -0.001217 20104.023438
2022-07-11 -0.064131 19970.556641
2022-07-19  0.031695 23389.433594
2022-07-27  0.134556 22930.548828
2022-08-02 -0.069805 22978.117188
2022-08-09  0.017999 23164.318359
2022-08-19  0.001132 20877.552734
2022-09-05  0.041133 19812.371094
2022-09-11  0.055493 21769.255859
2022-09-23  0.099090 19297.638672
2022-10-09  0.020161 19446.425781
2022-10-21  0.056775 19172.468750
2022-10-28 -0.086844 20595.351562
2022-11-07 -0.097668 20602.816406
2022-11-16  0.015232 16669.439453
2022-11-25 -0.017656 16521.841797
2022-12-03  0.006273 16908.236328
2022-12-12  0.028184 17206.437500
2022-12-22  0.130638 16830.341797
2023-01-10  0.042490 17446.292969
2023-01-20  0.026135 22676.552734
2023-01-30  0.128076 22840.138672
2023-02-04  0.070716 23331.847656
2023-02-15  0.076543 24307

# **Prepare Tweet Agents for NetLogo**

This cell generates the second CSV file required by the NetLogo
simulation — one that enables **tweet posts to function as
independent agents** rather than a single aggregated number.

**Why tweet agents?**

In a model that uses only the daily $S_t$ average, every investor
sees the exact same sentiment signal at the exact same time —
an unrealistic assumption. By making individual tweets into agents,
the model captures a more realistic dynamic: investors are exposed
to different tweets based on spatial proximity (vision radius),
and high-reach viral tweets influence more investors than
low-engagement posts.

**Selection method:**

For each of the 36 dates, the **top 10 most influential tweets**
are selected based on an influence score:

$$\text{influence}_i = (\text{likes}_i + \text{retweets}_i + 1) \times (\text{followers}_i + 1)$$

Reach and engagement values are then normalised globally to [0, 1]
so that influence is comparable across all dates.

**Output:** `outputs/tweet_agents.csv`
- 360 rows total (36 dates × 10 tweets per date)
- Columns: `tick`, `tweet_sentiment`, `reach`, `engagement`
- Loaded directly into NetLogo at simulation startup

In [29]:
# Selects the top 10 most influential tweets per date
# Output: outputs/tweet_agents.csv  (360 rows: 36 dates × 10 tweets)

# Load individually scored tweets
df_scored = pd.read_csv('outputs/tweets_scored.csv', parse_dates=['date'])

#  Compute influence score
# Influence = (likes + retweets + 1) × (followers + 1)
df_scored['engagement'] = df_scored['likes'] + df_scored['retweets'] + 1
df_scored['reach']      = df_scored['followers'] + 1
df_scored['influence']  = df_scored['engagement'] * df_scored['reach']

# Map each date to a tick number (1 to 36)
date_order  = sorted(df_scored['date'].dt.date.unique())
date_to_tick = {d: i + 1 for i, d in enumerate(date_order)}
df_scored['tick'] = df_scored['date'].dt.date.map(date_to_tick)

# Select top 10 tweets per date by influence
top10 = (
    df_scored
    .sort_values('influence', ascending=False)
    .groupby('tick')
    .head(10)
    .reset_index(drop=True)
)

print(f"Total tweet agents selected: {len(top10)}")
print(f"Tweets per tick:\n{top10.groupby('tick').size().to_string()}")

# Normalise reach and engagement globally to [0, 1]
top10['reach_norm']      = top10['reach']      / top10['reach'].max()
top10['engagement_norm'] = top10['engagement'] / top10['engagement'].max()

# Final output: only what NetLogo needs
tweet_agents = top10[[
    'tick',
    'finbert_score',      # individual tweet sentiment  St_tweet ∈ [-1,1]
    'reach_norm',         # how many investors this tweet can reach
    'engagement_norm',    # how viral / visible this tweet is
]].copy()

tweet_agents.columns = ['tick', 'tweet_sentiment', 'reach', 'engagement']

tweet_agents.to_csv('outputs/tweet_agents.csv', index=False)
print("\nSaved → outputs/tweet_agents.csv")
print("\nSample (top 5 rows):")
print(tweet_agents.head(10).to_string(index=False))

# Quick summary
print("\nSentiment range of tweet agents:")
print(f"  Most negative tweet:  {tweet_agents['tweet_sentiment'].min():.4f}")
print(f"  Most positive tweet:  {tweet_agents['tweet_sentiment'].max():.4f}")
print(f"  Mean sentiment:       {tweet_agents['tweet_sentiment'].mean():.4f}")

Total tweet agents selected: 360
Tweets per tick:
tick
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
24    10
25    10
26    10
27    10
28    10
29    10
30    10
31    10
32    10
33    10
34    10
35    10
36    10

Saved → outputs/tweet_agents.csv

Sample (top 5 rows):
 tick  tweet_sentiment    reach  engagement
   18        -0.174194 0.154394    0.233018
   15         0.068595 0.089477    0.337268
   17         0.286399 0.192389    0.128312
   33        -0.275281 0.022466    0.592321
    6         0.695213 0.055653    0.219188
    4         0.172247 0.192388    0.052885
   20         0.151758 0.058834    0.156628
   17         0.015156 0.055653    0.157388
   17         0.008467 0.027709    0.311129
   27         0.224127 0.192390    0.042754

Sentiment range of tweet agents:
  Most negative tweet:  -0.9640
  Most positive

#  **Final NetLogo CSV Intergration Check**

This section validates the final CSV files before they are imported into NetLogo. It checks whether the required columns are present, identifies missing values, verifies that sentiment and normalized influence variables remain within their expected ranges, and confirms that tick values are sequential and consistent across both datasets. Successful completion of these checks indicates that `daily_sentiment_with_btc.csv` and `tweet_agents.csv` are structurally compatible with the NetLogo model.


In [30]:
daily_file = 'outputs/daily_sentiment_with_btc.csv'
tweet_file = 'outputs/tweet_agents.csv'

required_daily_columns = [
    'date',
    'St',
    'tick',
    'tweet_count',
    'btc_close'
]

required_tweet_columns = [
    'tick',
    'tweet_sentiment',
    'reach',
    'engagement'
]

daily_check = pd.read_csv(daily_file)
tweet_check = pd.read_csv(tweet_file)

# Validate column structures
assert list(daily_check.columns) == required_daily_columns, (
    f"Unexpected daily CSV columns: {list(daily_check.columns)}"
)

assert list(tweet_check.columns) == required_tweet_columns, (
    f"Unexpected tweet-agent CSV columns: {list(tweet_check.columns)}"
)

# Validate missing values
assert daily_check[required_daily_columns].notna().all().all(), (
    "Missing values detected in daily_sentiment_with_btc.csv"
)

assert tweet_check[required_tweet_columns].notna().all().all(), (
    "Missing values detected in tweet_agents.csv"
)

# Validate valid ranges
assert daily_check['St'].between(-1, 1).all(), (
    "Daily sentiment values must remain within [-1, 1]"
)

assert tweet_check['tweet_sentiment'].between(-1, 1).all(), (
    "Tweet sentiment values must remain within [-1, 1]"
)

assert tweet_check['reach'].between(0, 1).all(), (
    "Reach values must remain within [0, 1]"
)

assert tweet_check['engagement'].between(0, 1).all(), (
    "Engagement values must remain within [0, 1]"
)

# Validate ticks
expected_daily_ticks = list(range(1, len(daily_check) + 1))

assert daily_check['tick'].tolist() == expected_daily_ticks, (
    "Daily tick values are not sequential."
)

assert set(tweet_check['tick']).issubset(set(daily_check['tick'])), (
    "Tweet-agent CSV contains ticks not available in the daily dataset."
)

print("=" * 60)
print("NETLOGO CSV INTEGRATION CHECK PASSED")
print("=" * 60)
print(f"Daily observations : {len(daily_check)}")
print(f"Tweet agents       : {len(tweet_check)}")
print(f"Date range         : {daily_check['date'].min()} to {daily_check['date'].max()}")
print(f"Daily sentiment    : {daily_check['St'].min():.4f} to {daily_check['St'].max():.4f}")
print(f"Tweet sentiment    : {tweet_check['tweet_sentiment'].min():.4f} to "
      f"{tweet_check['tweet_sentiment'].max():.4f}")
print("Both CSV files are structurally compatible with the NetLogo model.")

NETLOGO CSV INTEGRATION CHECK PASSED
Daily observations : 36
Tweet agents       : 360
Date range         : 2022-06-08 to 2023-05-28
Daily sentiment    : -0.1316 to 0.1578
Tweet sentiment    : -0.9640 to 0.9046
Both CSV files are structurally compatible with the NetLogo model.
